In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense

In [ ]:
# Load dataset
with open("text_data.txt", "r", encoding="utf-8") as file:
    text = file.read()

# Convert to lowercase
text = text.lower()

print("First 500 characters of dataset:\n")
print(text[:500])

First 500 characters of dataset:

*** start of the project gutenberg ebook 11 ***

[illustration]




alice’s adventures in wonderland

by lewis carroll

the millennium fulcrum edition 3.0

contents

 chapter i.     down the rabbit-hole
 chapter ii.    the pool of tears
 chapter iii.   a caucus-race and a long tale
 chapter iv.    the rabbit sends in a little bill
 chapter v.     advice from a caterpillar
 chapter vi.    pig and pepper
 chapter vii.   a mad tea-party
 chapter viii.  the queen’s croquet-ground
 chapter ix.    the


In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1

print("Total vocabulary size:", total_words)

Total vocabulary size: 3067


In [ ]:
input_sequences = []

for line in text.split('\n'):

    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

print("Total input sequences:", len(input_sequences))

Total input sequences: 25278


In [ ]:
max_seq_len = max([len(seq) for seq in input_sequences])

input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')
)

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print("Input shape:", X.shape)
print("Output shape:", y.shape)

Input shape: (25278, 17)
Output shape: (25278, 3067)


In [ ]:
rnn_model = Sequential()

rnn_model.add(Embedding(total_words, 100, input_length=max_seq_len-1))

rnn_model.add(SimpleRNN(150))

rnn_model.add(Dense(total_words, activation='softmax'))

rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

rnn_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
rnn_model.fit(
    X,
    y,
    epochs=50,
    verbose=1
)

Epoch 1/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 23s 25ms/step - accuracy: 0.0543 - loss: 6.4439
Epoch 2/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 19s 23ms/step - accuracy: 0.1018 - loss: 5.5380
Epoch 3/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 17s 21ms/step - accuracy: 0.1558 - loss: 4.9574
Epoch 4/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 21s 23ms/step - accuracy: 0.1901 - loss: 4.4306
Epoch 5/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 17s 21ms/step - accuracy: 0.2296 - loss: 3.9952
Epoch 6/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 17s 21ms/step - accuracy: 0.2765 - loss: 3.5354
Epoch 7/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 18s 23ms/step - accuracy: 0.3374 - loss: 3.1550
Epoch 8/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 17s 21ms/step - accuracy: 0.3967 - loss: 2.8158
Epoch 9/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 17s 21ms/step - accuracy: 0.4610 - loss: 2.4676
Epoch 10/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 20s 21ms/step - accuracy: 0.5140 - loss: 2.2026
Epoch 11/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 17s 21ms/step - accuracy: 0.5692 - loss: 1.9553
Epoch 12/50
790/790 ━━━━━━━━━━

In [ ]:
lstm_model = Sequential()

lstm_model.add(Embedding(total_words, 100, input_length=max_seq_len-1))

lstm_model.add(LSTM(150))

lstm_model.add(Dense(total_words, activation='softmax'))

lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
lstm_model.fit(
    X,
    y,
    epochs=50,
    verbose=1
)

Epoch 1/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 38s 44ms/step - accuracy: 0.0572 - loss: 6.4072
Epoch 2/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 34s 42ms/step - accuracy: 0.0911 - loss: 5.6668
Epoch 3/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 36s 45ms/step - accuracy: 0.1299 - loss: 5.2330
Epoch 4/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 35s 44ms/step - accuracy: 0.1658 - loss: 4.8458
Epoch 5/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 37s 47ms/step - accuracy: 0.1834 - loss: 4.5320
Epoch 6/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 35s 44ms/step - accuracy: 0.1975 - loss: 4.2884
Epoch 7/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 38s 48ms/step - accuracy: 0.2180 - loss: 4.0448
Epoch 8/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 36s 45ms/step - accuracy: 0.2338 - loss: 3.8228
Epoch 9/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 41s 46ms/step - accuracy: 0.2581 - loss: 3.6166
Epoch 10/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 36s 45ms/step - accuracy: 0.2776 - loss: 3.4111
Epoch 11/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 36s 45ms/step - accuracy: 0.3045 - loss: 3.2152
Epoch 12/50
790/790 ━━━━━━━━━━

In [ ]:
def generate_text(seed_text, next_words, model):

    for _ in range(next_words):

        token_list = tokenizer.texts_to_sequences([seed_text])[0]

        token_list = pad_sequences(
            [token_list],
            maxlen=max_seq_len-1,
            padding='pre'
        )

        predicted = np.argmax(model.predict(token_list), axis=-1)

        output_word = ""

        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break

        seed_text += " " + output_word

    return seed_text

In [ ]:
seed = "alice was"

generated_text_rnn = generate_text(seed, 10, rnn_model)

print("Generated Text using RNN:")
print(generated_text_rnn)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 416ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Generated Text using RNN:
alice was thoroughly puzzled “does the boots and shoes ” she repeated


In [ ]:
seed = "alice was"

generated_text_lstm = generate_text(seed, 10, lstm_model)

print("Generated Text using LSTM:")
print(generated_text_lstm)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 458ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
Generated Text using LSTM:
alice was thoroughly puzzled “does the boots and shoes ” she repeated
